# Working

**Request:** [T12] Who are my top performers?

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T12] Who are my top performers?
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAPt85klDxZxLrf9joBszSjA=
> 
> ## Sugges

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd

data = {
    'Rep': ['Alice', 'Ben', 'Chidi'],
    'Revenue': [412000, 298000, 355000],
    'GrowthPct': [4, 41, 12],
    'MarginPct': [18, 22, 39]
}
df = pd.DataFrame(data)

# Normalize metrics (min-max scaling)
for col in ['Revenue', 'GrowthPct', 'MarginPct']:
    min_val = df[col].min()
    max_val = df[col].max()
    if max_val == min_val:
        df[f'{col}_Normalized'] = 0.5 # Assign a neutral score if no variation
    else:
        df[f'{col}_Normalized'] = (df[col] - min_val) / (max_val - min_val)

# Calculate Performance Score (equal weighting for now)
df['Performance_Score'] = df['Revenue_Normalized'] + df['GrowthPct_Normalized'] + df['MarginPct_Normalized']

# Rank performers
df = df.sort_values(by='Performance_Score', ascending=False).reset_index(drop=True)
df['Rank'] = df.index + 1

# Prepare data for Excel output
output_df = df[['Rep', 'Revenue', 'GrowthPct', 'MarginPct', 'Performance_Score', 'Rank']]

# Create summary
top_performer = output_df.iloc[0]['Rep']
summary_text = f"Top Performer: {top_performer} with a Performance Score of {output_df.iloc[0]['Performance_Score']:.2f}.\n\n" \
               f"The performance score is calculated by normalizing Revenue, Growth Percentage, and Margin Percentage, then summing the normalized values. " \
               f"A higher score indicates better overall performance. The ranking is based on this combined score."


# Save to a temporary file (e.g., CSV for now, will be converted to Excel later if needed by the platform)
# Or, more directly, save a representation that can be used to build the Excel.
# For now, print relevant data to stdout to capture for the next step.

print("---Summary---")
print(summary_text)
print("---Data---")
print(output_df.to_json(orient='records'))


---Summary---
Top Performer: Chidi with a Performance Score of 1.72.

The performance score is calculated by normalizing Revenue, Growth Percentage, and Margin Percentage, then summing the normalized values. A higher score indicates better overall performance. The ranking is based on this combined score.
---Data---
[{"Rep":"Chidi","Revenue":355000,"GrowthPct":12,"MarginPct":39,"Performance_Score":1.7162162162,"Rank":1},{"Rep":"Ben","Revenue":298000,"GrowthPct":41,"MarginPct":22,"Performance_Score":1.1904761905,"Rank":2},{"Rep":"Alice","Revenue":412000,"GrowthPct":4,"MarginPct":18,"Performance_Score":1.0,"Rank":3}]


In [2]:
import pandas as pd

data_str = """Rep,Revenue,GrowthPct,MarginPct
Alice,412000,4,18
Ben,298000,41,22
Chidi,355000,12,39"""

df = pd.read_csv(pd.io.common.StringIO(data_str))

# Normalize the metrics (0-1 scaling)
df['Normalized_Revenue'] = (df['Revenue'] - df['Revenue'].min()) / (df['Revenue'].max() - df['Revenue'].min())
df['Normalized_GrowthPct'] = (df['GrowthPct'] - df['GrowthPct'].min()) / (df['GrowthPct'].max() - df['GrowthPct'].min())
df['Normalized_MarginPct'] = (df['MarginPct'] - df['MarginPct'].min()) / (df['MarginPct'].max() - df['MarginPct'].min())

# Calculate Performance Score (simple sum of normalized metrics)
df['Performance_Score'] = df['Normalized_Revenue'] + df['Normalized_GrowthPct'] + df['Normalized_MarginPct']

# Rank performers
df = df.sort_values(by='Performance_Score', ascending=False).reset_index(drop=True)
df['Rank'] = df.index + 1

# Select relevant columns for the final output
output_df = df[['Rep', 'Revenue', 'GrowthPct', 'MarginPct', 'Performance_Score', 'Rank']]

# Prepare summary string
top_performer = output_df.iloc[0]['Rep']
top_score = output_df.iloc[0]['Performance_Score']
summary = f"Top Performer: {top_performer} with a Performance Score of {top_score:.2f}.\n\nThe performance score is calculated by normalizing Revenue, Growth Percentage, and Margin Percentage, then summing the normalized values. A higher score indicates better overall performance. The ranking is based on this combined score."

# Create an Excel writer object
output_file = '/tmp/output/rep_performance_analysis.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')

# Write summary to a 'Summary' sheet
summary_df = pd.DataFrame([summary], columns=['Key Findings'])
summary_df.to_excel(writer, sheet_name='Summary', index=False)

# Write the full data to a 'Performance Data' sheet
output_df.to_excel(writer, sheet_name='Performance Data', index=False)

writer.close()

print(f"Summary:\n{summary}")
print(f"Output file: {output_file}")


Summary:
Top Performer: Chidi with a Performance Score of 1.72.

The performance score is calculated by normalizing Revenue, Growth Percentage, and Margin Percentage, then summing the normalized values. A higher score indicates better overall performance. The ranking is based on this combined score.
Output file: /tmp/output/rep_performance_analysis.xlsx



[files written: rep_performance_analysis.xlsx]
